## skeleton

In [ ]:

import json
import re
import unicodedata
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from datetime import datetime
import time
from collections import Counter
import pdfplumber
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm.auto import tqdm
import langid
import random
import spacy


class ClimateReportAnalyzer:
    # Chunking settings
    MIN_CHARS = 400
    MAX_CHARS = 2000
    MAX_TOKENS = MAX_CHARS // 4
    BATCH_SIZE = 48

    # Translation settings
    TRANSLATE_BATCH_SIZE = 48
    TRANSLATE_INPUT_MAX_TOKENS = 400
    TRANSLATE_OUTPUT_MAX_TOKENS = 400

    def __init__(self, cache_dir="cache"):
        # ...


### old

In [ ]:
 def calculate_report_score(self, analyzed_chunks: List[Dict]) -> Dict:
        """Calculate overall scores for the report"""
        # Correct labels for climate-specificity model
        score_map = {
            "specific": 1.0,
            "non-specific": 0.0,
        }

        specificity_scores = []
        for chunk in analyzed_chunks:
            label = chunk.get('specificity_label', '').lower()
            confidence = chunk.get('specificity_score', 0.5)

            # Map label to score
            base_score = score_map.get(label, 0.5)  # Default 0.5 if unknown
            weighted_score = base_score * confidence
            specificity_scores.append(weighted_score)

        overall_score = sum(specificity_scores) / len(specificity_scores) if specificity_scores else 0

        # Count labels
        label_dist = {}
        for chunk in analyzed_chunks:
            label = chunk.get('specificity_label', 'unknown')
            label_dist[label] = label_dist.get(label, 0) + 1

        return {
            'overall_specificity': overall_score,
            'num_chunks_analyzed': len(analyzed_chunks),
            'num_climate_chunks': len([c for c in analyzed_chunks if c.get('specificity_label') == 'specific']),
            'label_distribution': label_dist,
            'specificity_scores': specificity_scores
        }